# **Day 70: Threading (Doing Many Things Together, Part 1)**

Normally, Python runs one line after another. If one line has to **wait** (for example, for a file to download), everything else also waits.

**Threads** let a program do **many waiting jobs at the same time**.

**Real-life example:** A cook puts rice to boil and, while waiting, cuts vegetables. The cook does not stand and stare at the rice.

**Important:** Because of something called the **GIL** (Global Interpreter Lock), Python threads **do not** make heavy calculations faster. They help only when the program is **waiting**, for example for the internet or the disk. This kind of work is called **I/O-bound** work.

## **Example from Remote Sensing: Downloading Many Tiles**
Downloading satellite tiles means a lot of waiting for the internet. We will **pretend** to download with `time.sleep()`.

In [1]:
import threading, time

def download(tile, seconds):
    print(f"  Start {tile}")
    time.sleep(seconds)                # Pretend we are waiting for the internet
    print(f"  Done  {tile} ({seconds}s)")

tiles = [("45QXF", 1.0), ("45QXE", 1.2), ("45QYF", 0.8), ("45RXL", 1.1)]

start = time.perf_counter()
for t, s in tiles:
    download(t, s)
print(f"One by one: {time.perf_counter() - start:.2f} seconds")

  Start 45QXF


  Done  45QXF (1.0s)
  Start 45QXE


  Done  45QXE (1.2s)
  Start 45QYF


  Done  45QYF (0.8s)
  Start 45RXL


  Done  45RXL (1.1s)
One by one: 4.10 seconds


In [2]:
start = time.perf_counter()
threads = [threading.Thread(target=download, args=(t, s)) for t, s in tiles]
for th in threads:
    th.start()                         # Start all downloads
for th in threads:
    th.join()                          # Wait for all to finish
print(f"With threads: {time.perf_counter() - start:.2f} seconds")

  Start 45QXF
  Start 45QXE
  Start 45QYF
  Start 45RXL


  Done  45QYF (0.8s)
  Done  45QXF (1.0s)


  Done  45RXL (1.1s)
  Done  45QXE (1.2s)
With threads: 1.24 seconds


With threads, the total time was about the time of the **slowest** download, not the sum of all of them. The waiting happened **at the same time**.

## **ThreadPoolExecutor: The Easy Way**
`ThreadPoolExecutor` manages the threads for us.

In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def fake_download(tile):
    time.sleep(0.5)
    sizes = {"45QXF": 812, "45QXE": 790, "45QYF": 805, "45RXL": 776, "44QPM": 798}
    return tile, sizes[tile]

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(fake_download, ["45QXF", "45QXE", "45QYF", "45RXL", "44QPM"]))

for tile, mb in results:
    print(f"{tile}: {mb} MB")
print(f"Total time: {time.perf_counter() - start:.2f} seconds for 5 tiles")

45QXF: 812 MB
45QXE: 790 MB
45QYF: 805 MB
45RXL: 776 MB
44QPM: 798 MB
Total time: 0.51 seconds for 5 tiles


## **A Problem When Threads Share Data: Race Condition**
If many threads change the **same variable** at the same time, some changes can get lost. This is called a **race condition**.

In [4]:
counter = 0

def add_many(n):
    global counter
    for _ in range(n):
        temp = counter
        time.sleep(0)            # Gives other threads a chance to jump in here
        counter = temp + 1

threads = [threading.Thread(target=add_many, args=(1000,)) for _ in range(4)]
for th in threads: th.start()
for th in threads: th.join()
print("Expected 4000, got", counter)

Expected 4000, got 1699


Many updates were lost, because two threads read the same old value and both wrote back the same new value.

**The fix:** A **Lock**. Only one thread can hold the lock at a time, like a **single key to a washroom**.

In [5]:
counter = 0
lock = threading.Lock()

def add_many_safely(n):
    global counter
    for _ in range(n):
        with lock:
            temp = counter
            time.sleep(0)
            counter = temp + 1

threads = [threading.Thread(target=add_many_safely, args=(1000,)) for _ in range(4)]
for th in threads: th.start()
for th in threads: th.join()
print("Expected 4000, got", counter)

Expected 4000, got 4000


## **Threads Do Not Speed Up Heavy Calculations**

In [6]:
def heavy_work(n):
    return sum(i * i for i in range(n))

N = 3_000_000
start = time.perf_counter()
for _ in range(4):
    heavy_work(N)
one_by_one = time.perf_counter() - start

start = time.perf_counter()
with ThreadPoolExecutor(4) as ex:
    list(ex.map(heavy_work, [N] * 4))
with_threads = time.perf_counter() - start

print(f"One by one: {one_by_one:.2f}s, with threads: {with_threads:.2f}s")

One by one: 0.58s, with threads: 0.59s


There is no real speed-up for calculations. For heavy calculations, we need **multiprocessing**, which is tomorrow's topic.

In [7]:
# Quick question
# Which job will get faster with threads: (a) downloading 50 files, (b) calculating NDVI for a huge image?